# Feature Engineering and Evaluation Split

Features are derived from information available at loan approval. Deterministic feature engineering is performed before splitting; learned preprocessing steps—imputation, scaling, and one-hot encoding—are fit only on training data.

Evaluation uses:
- Training: FY2010–2017
- Validation: FY2018
- Final test set: FY2019

In [96]:
import pandas as pd
import numpy as np

CLEANED_PATH = "../data/processed/sba_7a_cleaned.csv"

df = pd.read_csv(
    CLEANED_PATH,
    dtype={"naics_sector": "string"}
)
df.shape

(428874, 24)

In [97]:
df["default"].value_counts(normalize=True).mul(100).round(2)

default
0    92.04
1     7.96
Name: proportion, dtype: float64

In [98]:
df.head()

,BorrState,BankState,GrossApproval,SBAGuaranteedApproval,ApprovalFY,ProcessingMethod,InitialInterestRate,FixedorVariableInterestInd,TermInMonths,NaicsCode,...,BusinessAge,RevolverStatus,JobsSupported,CollateralInd,default,sba_guarantee_ratio,is_franchise,naics_sector,approval_month,sold_secondary_market
0,TX,TX,288000.0,259200.0,2010,Preferred Lenders Program,6.00,V,120.0,722110.0,...,Unanswered,N,18.0,Y,0,0.9,0,72,10,NaN
1,FL,NC,1200000.0,1080000.0,2010,Preferred Lenders Program,4.75,V,300.0,541940.0,...,"Existing, 5 or more years",N,10.0,N,0,0.9,0,54,10,1.0
2,TX,OH,120000.0,108000.0,2010,Preferred Lenders Program,5.25,V,90.0,312113.0,...,Less than 4 years old but at least 3,N,4.0,N,0,0.9,0,31,10,NaN
3,MI,OH,150000.0,75000.0,2010,SBA Express Program,5.25,V,60.0,722211.0,...,"Startup, Loan Funds will Open Business",N,48.0,Y,0,0.5,1,72,10,NaN
4,NY,OH,25000.0,12500.0,2010,SBA Express Program,6.50,V,84.0,238210.0,...,"Existing, 5 or more years",Y,4.0,Y,0,0.5,0,23,10,NaN


In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 428874 entries, 0 to 428873
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   BorrState                   428874 non-null  str    
 1   BankState                   428874 non-null  str    
 2   GrossApproval               428874 non-null  float64
 3   SBAGuaranteedApproval       428874 non-null  float64
 4   ApprovalFY                  428874 non-null  int64  
 5   ProcessingMethod            428874 non-null  str    
 6   InitialInterestRate         428869 non-null  float64
 7   FixedorVariableInterestInd  428874 non-null  str    
 8   TermInMonths                428652 non-null  float64
 9   NaicsCode                   428872 non-null  float64
 10  ProjectState                428874 non-null  str    
 11  SBADistrictOffice           428874 non-null  str    
 12  CongressionalDistrict       428849 non-null  float64
 13  BusinessType             

In [100]:
df.isna().mean().mul(100).sort_values(ascending=False)

sold_secondary_market         75.565318
BusinessAge                    0.286798
TermInMonths                   0.051763
CongressionalDistrict          0.005829
BusinessType                   0.003964
InitialInterestRate            0.001166
NaicsCode                      0.000466
naics_sector                   0.000466
FixedorVariableInterestInd     0.000000
ProcessingMethod               0.000000
ApprovalFY                     0.000000
SBAGuaranteedApproval          0.000000
BankState                      0.000000
GrossApproval                  0.000000
BorrState                      0.000000
ProjectState                   0.000000
RevolverStatus                 0.000000
SBADistrictOffice              0.000000
JobsSupported                  0.000000
CollateralInd                  0.000000
sba_guarantee_ratio            0.000000
default                        0.000000
is_franchise                   0.000000
approval_month                 0.000000
dtype: float64

In [101]:
df["log_gross_approval"] = np.log1p(df["GrossApproval"])
df["log_sba_guaranteed_approval"] = np.log1p(df["SBAGuaranteedApproval"])

In [102]:
df["term_years"] = df["TermInMonths"] / 12

In [103]:
df["guarantee_ratio_bucket"] = pd.cut(
    df["sba_guarantee_ratio"],
    bins=[0, 0.5, 0.75, 0.85, 1.0],
    labels=["low", "medium", "high", "very_high"],
    include_lowest=True
)

In [104]:
df["same_state_lender"] = (df["BorrState"] == df["BankState"]).astype(int)

In [105]:
state_match_rate = (df['BorrState'] == df['ProjectState']).mean()
print(f'BorrState and ProjectState match in {state_match_rate:.2%} of rows.')

BorrState and ProjectState match in 99.78% of rows.


In [106]:
business_age_map = {
    "Existing, 5 or more years": "existing_5_plus",
    "Existing or more than 2 years old": "existing_2_plus",
    "New, Less than 1 Year old": "new_under_1",
    "New Business or 2 years or less": "new_2_or_less",
    "Startup, Loan Funds will Open Business": "startup",
    "Change of Ownership": "change_ownership",
    "Unanswered": "unknown",
    "Less than 2 years old but at least 1": "young_1_to_2",
    "Less than 3 years old but at least 2": "young_2_to_3",
    "Less than 4 years old but at least 3": "young_3_to_4",
    "Less than 5 years old but at least 4": "young_4_to_5",
    "Loan Funds will Open Business": "startup",
}

unmapped_business_age = (
    set(df["BusinessAge"].dropna().unique())
    - set(business_age_map)
)

assert not unmapped_business_age, (
    f"Unmapped BusinessAge values: {unmapped_business_age}"
)

df["business_age_group"] = df["BusinessAge"].map(business_age_map).fillna("unknown")

In [107]:
df["naics_sector"] = df["naics_sector"].fillna("unknown")

In [108]:
df["CongressionalDistrict"] = (
    df["CongressionalDistrict"]
    .astype("Int64")
    .astype("string")
    .fillna("unknown")
)

df["approval_month"] = df["approval_month"].astype("string")

In [109]:
target_col = 'default'

# Drop raw or redundant columns after creating engineered features.
# BorrState is removed because ProjectState already captures the project location,
# and same_state_lender keeps the borrower/lender relationship signal.
drop_after_engineering = [
    'GrossApproval',
    'SBAGuaranteedApproval',
    'TermInMonths',
    'NaicsCode',
    'BusinessAge',
    'BorrState'
]

model_df = df.drop(columns=drop_after_engineering).copy()

model_df.shape

(428874, 24)

In [110]:
X = model_df.drop(columns=[target_col])
y = model_df[target_col]

X.shape, y.shape

((428874, 23), (428874,))

In [111]:
numeric_features = X.select_dtypes(include=["int64", "int32", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Number of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Numeric features: ['ApprovalFY', 'InitialInterestRate', 'JobsSupported', 'sba_guarantee_ratio', 'is_franchise', 'sold_secondary_market', 'log_gross_approval', 'log_sba_guaranteed_approval', 'term_years', 'same_state_lender']
Categorical features: ['BankState', 'ProcessingMethod', 'FixedorVariableInterestInd', 'ProjectState', 'SBADistrictOffice', 'CongressionalDistrict', 'BusinessType', 'RevolverStatus', 'CollateralInd', 'naics_sector', 'approval_month', 'guarantee_ratio_bucket', 'business_age_group']
Number of numeric features: 10
Number of categorical features: 13


## Preprocessing policy

Imputation, scaling, and one-hot encoding are learned from the training period only.

- Numeric features use median imputation, with missing-value indicators.
- Categorical features use most-frequent-value imputation and one-hot encoding.
- `handle_unknown="ignore"` allows later-year categories not seen in training to be processed safely.
- Sparse encoding is retained to keep memory usage reasonable for the large dataset.

In [112]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
    )),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created.")

Preprocessor created.


In [113]:
TRAIN_END_YEAR = 2017
VALIDATION_YEAR = 2018
TEST_YEAR = 2019

train_mask = model_df["ApprovalFY"] <= TRAIN_END_YEAR
validation_mask = model_df["ApprovalFY"] == VALIDATION_YEAR
test_mask = model_df["ApprovalFY"] == TEST_YEAR

assert (train_mask | validation_mask | test_mask).all()
assert not (train_mask & validation_mask).any()
assert not (train_mask & test_mask).any()
assert not (validation_mask & test_mask).any()

X_train = model_df.loc[train_mask].drop(columns=[target_col])
y_train = model_df.loc[train_mask, target_col]

X_validation = model_df.loc[validation_mask].drop(columns=[target_col])
y_validation = model_df.loc[validation_mask, target_col]

X_test = model_df.loc[test_mask].drop(columns=[target_col])
y_test = model_df.loc[test_mask, target_col]

for split_name, labels in {
    "Train (2010–2017)": y_train,
    "Validation (2018)": y_validation,
    "Test (2019)": y_test,
}.items():
    print(
        f"{split_name}: {len(labels):,} loans | "
        f"default rate: {labels.mean():.2%}"
    )

Train (2010–2017): 355,031 loans | default rate: 7.55%
Validation (2018): 42,237 loans | default rate: 10.13%
Test (2019): 31,606 loans | default rate: 9.67%


In [114]:
X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Validation shape:", X_validation_processed.shape)
print("Test shape:", X_test_processed.shape)

Train shape: (355031, 326)
Validation shape: (42237, 326)
Test shape: (31606, 326)


In [115]:
feature_names = preprocessor.get_feature_names_out()

len(feature_names), feature_names[:20]

(326,
 array(['num__ApprovalFY', 'num__InitialInterestRate',
        'num__JobsSupported', 'num__sba_guarantee_ratio',
        'num__is_franchise', 'num__sold_secondary_market',
        'num__log_gross_approval', 'num__log_sba_guaranteed_approval',
        'num__term_years', 'num__same_state_lender',
        'num__missingindicator_InitialInterestRate',
        'num__missingindicator_sold_secondary_market',
        'num__missingindicator_term_years', 'cat__BankState_AK',
        'cat__BankState_AL', 'cat__BankState_AR', 'cat__BankState_AZ',
        'cat__BankState_CA', 'cat__BankState_CO', 'cat__BankState_CT'],
       dtype=object))

In [116]:
from pathlib import Path

FEATURES_PATH = "../data/processed/sba_7a_features.csv"

Path("../data/processed").mkdir(parents=True, exist_ok=True)
model_df.to_csv(FEATURES_PATH, index=False)

In [117]:
reloaded_features = pd.read_csv(
    FEATURES_PATH,
    dtype={
        "naics_sector": "string",
        "CongressionalDistrict": "string",
        "approval_month": "string",
    },
    low_memory=False,
)

assert reloaded_features.shape == model_df.shape
assert reloaded_features.columns.tolist() == model_df.columns.tolist()

print("Feature dataset saved and reloaded successfully.")

Feature dataset saved and reloaded successfully.


The saved feature dataset contains only deterministic transformations. It does not contain imputed, scaled, or one-hot-encoded data because those transformations must be learned from the training period only.

## Feature Engineering Decisions

### Feature creation

Created deterministic approval-time features:

- `log_gross_approval` and `log_sba_guaranteed_approval` to reduce skew in loan amounts.
- `term_years` as a more interpretable version of loan term.
- `guarantee_ratio_bucket` to represent broad SBA-guarantee levels.
- `same_state_lender` to capture whether the borrower and lender are located in the same state.
- `business_age_group` to simplify the detailed business-age categories.
- `naics_sector` as a broad industry category derived from NAICS.
- `CongressionalDistrict` and `approval_month` are treated as categorical variables.

### Raw features removed

- Replaced raw loan amounts with their log-transformed versions.
- Replaced `TermInMonths` with `term_years`.
- Replaced raw `BusinessAge` with `business_age_group`.
- Replaced raw `NaicsCode` with `naics_sector`.
- Removed `BorrState` because `ProjectState` captures project location and `same_state_lender` retains the borrower-lender geographic relationship.

### Preprocessing and evaluation

- Numeric features use median imputation, missing-value indicators, and standard scaling.
- Categorical features use most-frequent-value imputation and one-hot encoding.
- All learned preprocessing is fit only on FY2010-2017 training data.
- Evaluation is chronological:
  - Training: FY2010-2017
  - Validation: FY2018
  - Final test set: FY2019

The saved feature dataset contains only deterministic transformations. Imputation, scaling, and one-hot encoding remain inside the modeling pipeline to prevent data leakage.